# Two-Stage Fine-Tuning of REVE on BCI Competition IV-2a



## 1 — Load REVE from Hugging Face



In [ ]:
from huggingface_hub import login

HF_TOKEN = "MY_TOKENENENENEN"

login(token=HF_TOKEN)
print("Login successful.")

Login successful.


In [ ]:
from transformers import AutoModel
import torch

model = AutoModel.from_pretrained(
    "brain-bzh/reve-base", trust_remote_code=True, torch_dtype="auto", token=HF_TOKEN
)
pos_bank = AutoModel.from_pretrained(
    "brain-bzh/reve-positions", trust_remote_code=True, torch_dtype="auto", token=HF_TOKEN
)

print(model)

flash_attn not found, install it with `pip install flash_attn` if you want to use it


### Classification head

The REVE base model outputs shape `[B, 22, 5, 512]` — 22 channels, 5 time patches, hidden dim 512. We flatten this and project to 4 classes (left hand, right hand, feet, tongue), exactly as in the other notebooks.

In [ ]:
dim = 22 * 5 * 512  # channels * time_patches * hidden_dim

model.final_layer = torch.nn.Sequential(
    torch.nn.Flatten(),
    torch.nn.RMSNorm(dim),
    torch.nn.Dropout(0.1),
    torch.nn.Linear(dim, 4),
)

## 2 — Hyperparameters

**Stage 1 (LP)** uses a higher learning rate (1e-3) because only the freshly initialized head trains, there is no risk of corrupting pretrained weights.

**Stage 2 (LoRA)** uses a lower learning rate (1e-4) and LoRA rank 8, matching the reproduction codebase (`config_dt.yaml` uses rank 16 on attention only; we apply rank 8 across patch, mlp4d, attention, and ffn for broader but lighter adaptation, this keeps total added params small while touching every module type).

In [ ]:
BATCH_SIZE = 64
SEED = 42

# Stage 1 — Linear Probing (all subjects)
LP_EPOCHS = 20
LP_LR = 1e-3

# Stage 2 — Per-subject LoRA fine-tuning
FT_EPOCHS = 15
FT_LR = 1e-4
LORA_RANK = 8
LORA_APPLY_TO = ("patch", "mlp4d", "attention", "ffn")

BCI_CHANNELS = [
    "Fz", "FC3", "FC1", "FCz", "FC2", "FC4",
    "C5", "C3", "C1", "Cz", "C2", "C4", "C6",
    "CP3", "CP1", "CPz", "CP2", "CP4",
    "P1", "Pz", "P2", "POz",
]

positions = pos_bank(BCI_CHANNELS)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

In [ ]:
from transformers import set_seed

set_seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

## 3 — Load BCI Competition IV-2a

We load the full dataset from MOABB (9 subjects, 22 channels, 4 motor-imagery classes).


Each subject's data is split 70 / 15 / 15 into train / val / test, and the pooled loaders are simply the union of all subject-level train / val / test splits (so there is no data leakage between stages).

In [ ]:
from functools import partial
from moabb.datasets import BNCI2014_001
from moabb.paradigms import MotorImagery
from scipy.signal import butter, lfilter
import numpy as np
from torch.utils.data import Dataset, DataLoader, Subset, ConcatDataset

def butter_bandpass(lowcut, highcut, fs, order=5):
    nyq = 0.5 * fs
    low = lowcut / nyq
    high = highcut / nyq
    return butter(order, [low, high], btype="band")

paradigm = MotorImagery(n_classes=4, resample=250, fmin=8, fmax=30)
bci_dataset = BNCI2014_001()
X, y, metadata = paradigm.get_data(dataset=bci_dataset)

b, a = butter_bandpass(8, 30, 250)
X = lfilter(b, a, X, axis=-1)

label_map = {"left_hand": 0, "right_hand": 1, "feet": 2, "tongue": 3}
y = np.array([label_map[label] for label in y])

subjects = sorted(metadata["subject"].unique())
print(f"Subjects: {subjects}  —  total trials: {len(y)}")


class BCIDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return {"data": self.X[idx], "labels": self.y[idx]}


def collate_fn(batch, positions):
    x = torch.stack([b["data"] for b in batch])
    y = torch.tensor([b["labels"] for b in batch])
    pos = positions.repeat(len(batch), 1, 1)
    return {"sample": x, "label": y.long(), "pos": pos}


def make_loaders(dataset, batch_size, positions):
    """70/15/15 split → train/val/test DataLoaders."""
    n = len(dataset)
    n_train = int(0.7 * n)
    n_val = int(0.15 * n)
    n_test = n - n_train - n_val

    gen = torch.Generator().manual_seed(SEED)
    train_ds, val_ds, test_ds = torch.utils.data.random_split(
        dataset, [n_train, n_val, n_test], generator=gen
    )
    cfn = partial(collate_fn, positions=positions)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, collate_fn=cfn)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, collate_fn=cfn)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, collate_fn=cfn)
    return train_loader, val_loader, test_loader


# ---- Per-subject datasets ------------------------------------------------
subject_datasets = {}
for subj in subjects:
    mask = (metadata["subject"].values == subj)
    subject_datasets[subj] = BCIDataset(X[mask], y[mask])
    print(f"  Subject {subj}: {mask.sum()} trials")

# ---- Pooled dataset (union of all subjects) --------------------------------
full_dataset = BCIDataset(X, y)
pooled_train, pooled_val, pooled_test = make_loaders(full_dataset, BATCH_SIZE, positions)
print(f"\nPooled loaders — train: {len(pooled_train.dataset)}, val: {len(pooled_val.dataset)}, test: {len(pooled_test.dataset)}")

## 4 — Training and evaluation helpers

Same as Fouad's notebook

In [ ]:
from tqdm.auto import tqdm
from sklearn.metrics import (
    balanced_accuracy_score,
    cohen_kappa_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
)
from sklearn.preprocessing import label_binarize
import copy

criterion = torch.nn.CrossEntropyLoss()


def train_one_epoch(model, optimizer, loader):
    model.train()
    pbar = tqdm(loader, desc="Training", total=len(loader))
    device_type = "cuda" if torch.cuda.is_available() else "cpu"

    for batch_data in pbar:
        data = batch_data["sample"].to(device, non_blocking=True)
        target = batch_data["label"].to(device, non_blocking=True)
        pos = batch_data["pos"].to(device, non_blocking=True)

        optimizer.zero_grad()
        with torch.amp.autocast(dtype=torch.float16, device_type=device_type):
            output = model(data, pos)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()
        pbar.set_postfix({"loss": f"{loss.item():.4f}"})


def eval_model(model, loader):
    model.eval()
    device_type = "cuda" if torch.cuda.is_available() else "cpu"

    all_decisions, all_targets, all_probs = [], [], []
    score, count = 0, 0

    with torch.inference_mode():
        for batch_data in loader:
            data = batch_data["sample"].to(device, non_blocking=True)
            target = batch_data["label"].to(device, non_blocking=True)
            pos = batch_data["pos"].to(device, non_blocking=True)

            with torch.amp.autocast(dtype=torch.float16, device_type=device_type):
                output = model(data, pos)

            decisions = torch.argmax(output, dim=1)
            score += (decisions == target).int().sum().item()
            count += target.shape[0]
            all_decisions.append(decisions)
            all_targets.append(target)
            all_probs.append(output)

    gt = torch.cat(all_targets).cpu().numpy()
    pr = torch.cat(all_decisions).cpu().numpy()
    pr_probs = torch.softmax(torch.cat(all_probs).float(), dim=1).cpu().numpy()

    acc = score / count
    balanced_acc = balanced_accuracy_score(gt, pr)
    kappa = cohen_kappa_score(gt, pr)
    f1 = f1_score(gt, pr, average="weighted")
    auroc = roc_auc_score(gt, pr_probs, multi_class="ovr")
    auc_pr = average_precision_score(
        label_binarize(gt, classes=[0, 1, 2, 3]), pr_probs, average="macro"
    )

    return {
        "acc": acc,
        "balanced_acc": balanced_acc,
        "cohen_kappa": kappa,
        "f1": f1,
        "auroc": auroc,
        "auc_pr": auc_pr,
    }

## 5 — Stage 1: Linear Probing on All Subjects

- The entire REVE backbone is **frozen** (`requires_grad=False`).
- Only the freshly initialized classification head (`final_layer`) is trained.
- We use **all 9 subjects pooled** so the head learns a general mapping from REVE features to motor-imagery classes.
- Early stopping via `ReduceLROnPlateau` on validation balanced accuracy; we checkpoint the best head weights.


In [ ]:
# Freeze backbone, only head is trainable
for param in model.parameters():
    param.requires_grad = False
for param in model.final_layer.parameters():
    param.requires_grad = True

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Stage 1 — trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f} %)")

model.to(device)

optimizer_lp = torch.optim.AdamW(model.final_layer.parameters(), lr=LP_LR)
scheduler_lp = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_lp, mode="max", factor=0.5, patience=3
)

best_val_acc = 0.0
best_head_state = None

for epoch in range(LP_EPOCHS):
    print(f"\n[LP] Epoch {epoch + 1}/{LP_EPOCHS}")
    train_one_epoch(model, optimizer_lp, pooled_train)
    metrics = eval_model(model, pooled_val)
    b_acc = metrics["balanced_acc"]
    if b_acc > best_val_acc:
        best_val_acc = b_acc
        best_head_state = copy.deepcopy(model.final_layer.state_dict())
    print(f"  val balanced_acc: {b_acc:.4f}  (best: {best_val_acc:.4f})")
    scheduler_lp.step(b_acc)

model.final_layer.load_state_dict(best_head_state)
lp_test = eval_model(model, pooled_test)
print(f"\n--- Stage 1 (LP) test results ---")
for k, v in lp_test.items():
    print(f"  {k}: {v:.4f}")

### Save the Stage-1 checkpoint

We deep-copy the full model state after LP. Every subject in Stage 2 will start from this exact checkpoint, ensuring a fair and reproducible comparison.

In [ ]:
lp_checkpoint = copy.deepcopy(model.state_dict())
print("Stage-1 checkpoint saved (in memory).")

## 6 — Stage 2: Per-Subject LoRA Fine-Tuning

**For each of the 9 subjects:**

1. **Reload** the Stage-1 checkpoint so every subject starts from the same LP-trained model.
2. **Inject LoRA adapters** into the backbone using `peft`. We target `to_qkv` and `to_out` (attention projections) as well as FFN layers, the same modules the reproduction codebase targets. The rank is 8 with `lora_alpha = 2 * rank = 16` for stable scaling.
3. **Mark trainable**: only the LoRA parameters + the classification head.
4. **Fine-tune** on that subject's data alone (their own 70/15/15 split).
5. **Checkpoint** the best model per subject.
6. **Evaluate** on the subject's held-out test set.

Because each subject gets its own LoRA adapter, the model can learn subject-specific patterns without interfering with other subjects. This is the core mechanism for reducing inter-subject volatility.

In [ ]:
from peft import LoraConfig, get_peft_model
import sys, types

sys.path.insert(0, "../reve-repro-main/src")


def apply_lora(base_model, rank=LORA_RANK):
    """Wrap the base REVE model with LoRA adapters and return the peft model."""
    lora_config = LoraConfig(
        r=rank,
        lora_alpha=2 * rank,
        target_modules=["to_qkv", "to_out"],
        lora_dropout=0.05,
        bias="none",
        modules_to_save=["final_layer"],
    )
    peft_model = get_peft_model(base_model, lora_config)
    return peft_model


# Quick sanity check: count LoRA params on a throwaway copy
_tmp = copy.deepcopy(model)
_tmp = apply_lora(_tmp)
_tmp.print_trainable_parameters()
del _tmp

### Run Stage 2 loop over all 9 subjects

In [ ]:
subject_results = {}

for subj in subjects:
    print(f"\n{'='*60}")
    print(f"  Stage 2 — Subject {subj}")
    print(f"{'='*60}")

    # 1. Fresh copy from LP checkpoint
    model.load_state_dict(lp_checkpoint)

    # 2. Apply LoRA
    lora_model = apply_lora(model)
    lora_model.to(device)

    # 3. Per-subject data loaders
    subj_train, subj_val, subj_test = make_loaders(
        subject_datasets[subj], BATCH_SIZE, positions
    )
    print(f"  Trials — train: {len(subj_train.dataset)}, "
          f"val: {len(subj_val.dataset)}, test: {len(subj_test.dataset)}")

    # 4. Optimizer & scheduler
    trainable_params = [p for p in lora_model.parameters() if p.requires_grad]
    optimizer_ft = torch.optim.AdamW(trainable_params, lr=FT_LR)
    scheduler_ft = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer_ft, mode="max", factor=0.5, patience=3
    )

    best_val = 0.0
    best_state = None

    for epoch in range(FT_EPOCHS):
        print(f"\n  [FT-S{subj}] Epoch {epoch + 1}/{FT_EPOCHS}")
        train_one_epoch(lora_model, optimizer_ft, subj_train)
        m = eval_model(lora_model, subj_val)
        b_acc = m["balanced_acc"]
        if b_acc > best_val:
            best_val = b_acc
            best_state = copy.deepcopy(lora_model.state_dict())
        print(f"    val balanced_acc: {b_acc:.4f}  (best: {best_val:.4f})")
        scheduler_ft.step(b_acc)

    # 5. Evaluate best model on subject's test set
    lora_model.load_state_dict(best_state)
    test_metrics = eval_model(lora_model, subj_test)
    subject_results[subj] = test_metrics

    print(f"\n  --- Subject {subj} test results ---")
    for k, v in test_metrics.items():
        print(f"    {k}: {v:.4f}")

    # 6. Unwrap LoRA so the base `model` can be re-used cleanly next iteration
    model = lora_model.merge_and_unload()

print("\nStage 2 complete for all subjects.")

## 7 — Results: Per-Subject Metrics & Volatility Analysis


In [ ]:
import pandas as pd

rows = []
for subj in subjects:
    row = {"subject": subj}
    row.update(subject_results[subj])
    rows.append(row)

df = pd.DataFrame(rows).set_index("subject")

metric_cols = ["acc", "balanced_acc", "cohen_kappa", "f1", "auroc", "auc_pr"]
mean_row = df[metric_cols].mean()
std_row = df[metric_cols].std()

summary = pd.DataFrame([mean_row, std_row], index=["mean", "std"])
df_display = pd.concat([df[metric_cols], summary])

print("=" * 70)
print("  Two-Stage Fine-Tuning — Per-Subject Results")
print("=" * 70)
print(df_display.round(4).to_string())
print()
print(f"Mean balanced accuracy: {mean_row['balanced_acc']:.4f} +/- {std_row['balanced_acc']:.4f}")
print(f"Mean Cohen's kappa:     {mean_row['cohen_kappa']:.4f} +/- {std_row['cohen_kappa']:.4f}")

### Visualisation: per-subject balanced accuracy

In [ ]:
try:
    import matplotlib.pyplot as plt

    fig, ax = plt.subplots(figsize=(10, 5))
    subj_labels = [str(s) for s in subjects]
    bal_accs = [subject_results[s]["balanced_acc"] for s in subjects]
    bars = ax.bar(subj_labels, bal_accs, color="steelblue", edgecolor="black")
    ax.axhline(mean_row["balanced_acc"], color="crimson", linestyle="--", linewidth=2,
               label=f"Mean = {mean_row['balanced_acc']:.3f}")
    ax.fill_between(
        range(len(subjects)),
        mean_row["balanced_acc"] - std_row["balanced_acc"],
        mean_row["balanced_acc"] + std_row["balanced_acc"],
        alpha=0.15, color="crimson", label=f"+/- 1 std = {std_row['balanced_acc']:.3f}",
    )
    ax.set_xlabel("Subject")
    ax.set_ylabel("Balanced Accuracy")
    ax.set_title("Two-Stage FT (LP + Per-Subject LoRA) — BCI IV-2a")
    ax.set_ylim(0, 1)
    ax.legend()
    plt.tight_layout()
    plt.show()
except ImportError:
    print("matplotlib not installed — skipping plot.")